# Models

- [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W4/M1/01_models.ipynb)

## Environment Setup

### Install dependencies

#### A. on Colab?

In [95]:
%pip install -q langchain langchain_openai langchain_community

#### B. Locally?

If using `uv` locally you can `uv sync` (provided you have the `pyproject.toml`).

### Environment variables

The following environment variables are used in this notebook:

| Variable               | Required | Purpose                                  |
|------------------------|----------|------------------------------------------|
| `OPENROUTER_API_KEY`   | Yes      | Authenticates requests to OpenRouter LLMs |

#### On Colab?

Store these as Colab **Secrets** (key icon in the left sidebar), using the variable names above (for example `OPENROUTER_API_KEY`). Grant this notebook access to the secrets so they can be read securely at runtime.

#### Locally?

Store these in a `.env` file in your project directory (which should be listed in `.gitignore` so it is never committed). The code below uses `python-dotenv` to load values from this file into environment variables at runtime.


::: {.callout-warning}

API Keys are secrets and thus [**shall never be exposed**](./never_expose_api_keys.qmd).

:::

### Reading environment variables in

In [96]:
import os

In [97]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

## OpenRouter

[OpenRouter](https://openrouter.ai/) is the Unified Interface For LLMs.

Key benefits include:

- One API for all models
    - no subscription to each provider needed
    - switch easily between models and providers by changing a `str` value
- Some models are free
- Circumvent regional restrictions

If you don't already have an OpenRouter API key, you can create one for free at: [OpenRouter](https://openrouter.ai/keys).

![Open Router](https://github.com/HassanAlgoz/B5/blob/main/content/W4/M1/assets/open_router.png?raw=1)

### Basic usage

Models can be utilized in two ways:

1. **Standalone** - Models can be called directly (outside of the agent loop) for tasks like text generation, classification, or extraction without the need for an agent framework.
2. **With agents** - Models can be dynamically specified when creating an [agent](/oss/python/langchain/agents#model) (*discussed later*).

LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). By default, the course will use  [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) because it is both popular and performant.

In [98]:
from langchain_openai import ChatOpenAI

In [99]:
# https://openrouter.ai/openai/gpt-5-nano
model_gpt5_nano = ChatOpenAI(
    model="openai/gpt-5-nano",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

### Choosing between models

The **quality**, **latency** and **price** triangle vary between models.

Below are some tools to help you select the best one yourself:

- Humans vote for the best model on websites like [Arena.ai](https://arena.ai/) using a blind approach to maintain fairness.
- You can also compare pricing, providers, and usage tends on [OpenRouter.ai](https://openrouter.ai/models)
- An automatic short list for models based on your priorities: [Model Recommendation | Artficial Analysis](https://artificialanalysis.ai/models/recommend)


#### Arabic models

- [Open Arabic LLM Leaderboard | OALL](https://huggingface.co/OALL)
- [QIMMA ⛰ قمة Leaderboard](https://huggingface.co/spaces/qimma/leaderboard)
- [Arabic LLM Leaderboard | Silma](https://silma.ai/arabic-llm-leaderboard)
- [مؤشر نضج الذكاء الاصطناعي | Balsam](https://benchmarks.ksaa.gov.sa/b/balsam?version=2)

### Model Parameters

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `max_tokens`: limits the total number of tokens in the response, effectively controlling how long the output can be.
* `temperature`: the sampling temperature
    - **Low temperature** (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses.
    - **High temperature** (close to 1) is good for creative tasks or generating varied responses.

![](https://github.com/HassanAlgoz/B5/blob/main/content/W4/M1/assets/temperature_1.png?raw=1)

Filtering for **Free** models, we find [NVIDIA's Nemotron-3](https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free):

In [107]:
# Low Temperature (0)
model_nemotron3_nano_precise = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# High Temperature (0.9)
model_nemotron3_nano_creative = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.9,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

::: {.callout-note}

### Aside: Running a model locally

LangChain supports running models locally on your own hardware. This is useful for scenarios where either data privacy is critical, you want to invoke a custom model, or when you want to avoid the costs incurred when using a cloud-based model.

[Ollama](https://docs.langchain.com/oss/python/integrations/chat/ollama) is one of the easiest ways to run chat and embedding models locally.

On commodity laptops (for example 16–32 GB RAM machines), good starting points for **tool-calling** models in Ollama include:

- `lfm2.5-thinking:1.2b` – very lightweight, designed specifically for on-device deployment.
- `granite4:3b` (or `granite4:1b` / `granite4:350m`) – strong instruction-following with built-in tool-calling support.
- `ministral-3:3b` – edge-optimized vision + text model with native function calling and agentic behavior.

You can always browse the [Ollama model catalog](https://ollama.com/models) and filter by the **Tools** tag and smaller parameter sizes (≤3B) to find up-to-date options that run well locally.

:::

## Models' Key Methods

Models subclass [Runnable](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable) which has these key methods:

1. [`.invoke()`](https://docs.langchain.com/oss/python/langchain/models#invocation): results are only sent from the server when generation stops
2. [`.stream()`](https://docs.langchain.com/oss/python/langchain/models#stream): results are sent from the server as they are being generated
3. [`.batch()`](https://docs.langchain.com/oss/python/langchain/models#batch): send multiple inputs at once

[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models

```{mermaid}
classDiagram
     class ChatModel {
         +invoke(input)
         +stream(input)
         +batch(inputs)
     }
     class AIMessage {
         +content
         +metadata
         +role
     }
     ChatModel "1" --> "1" AIMessage : returns
```

### 1. Invoke

The most straightforward way to call a model is to use `invoke()` with a single message or a list of messages:

In [101]:
message = model_nemotron3_nano_precise.invoke("what is Ramadan? make it super short")

.. this returns an `AIMessage` object:

In [ ]:
message

langchain_core.messages.ai.AIMessage

.. which has a `content` property, which includes the generated response text:

In [102]:
print(message.content)

Ramadan is the Islamic holy month during which Muslims fast from sunrise to sunset, pray more, and focus on spiritual reflection and charity. It ends with the celebration of Eid al‑Fitr.


## Messages

**Messages** are the fundamental unit of context for models in LangChain. They represent **the input and output of models**, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

*Messages* are objects that contain three things:

- **Content**: Actual model response: text, images, audio, documents, etc.
- **Metadata**: Optional fields such as response information, message IDs, and token usage
- **Role**: Identifies the message type. One of:
    1. [`SystemMessage`](https://docs.langchain.com/oss/python/langchain/messages#system-message): Tells the model how to behave and provide context for interactions
    2. [`HumanMessage`](https://docs.langchain.com/oss/python/langchain/messages#human-message): Represents user input and interactions with the model
    3. [`AIMessage`](https://docs.langchain.com/oss/python/langchain/messages#ai-message): Responses generated by the model, including text content, tool calls, and metadata
    4. [`ToolMessage`](https://docs.langchain.com/oss/python/langchain/messages#tool-message): Represents the outputs of [tool calls](https://docs.langchain.com/oss/python/langchain/models#tool-calling)


In [103]:
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

In [104]:
system_prompt = SystemMessage("You always answer with 10 concise words, no more.")
user_prompt = HumanMessage("How do I make an Agent in Python?")

messages = [
    system_prompt,
    user_prompt,
]
response = model_nemotron3_nano_precise.invoke(messages)

In [105]:
print(response.content)

Define class, inherit from base, implement act, sense, communicate effectively.


### Metadata

An `AIMessage` can hold token counts and other usage metadata in its [`usage_metadata`](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata) field:

In [ ]:
response.usage_metadata

{'input_tokens': 38,
 'output_tokens': 299,
 'total_tokens': 337,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 276}}

### Conversation

A list of messages can be provided to a chat model to represent conversation history. Each message has a role that models use to indicate who sent the message in the conversation.



In [ ]:
conversation = [
    SystemMessage("You are a helpful assistant that translates English to Arabic."),
    HumanMessage("Translate: I love programming."),
    AIMessage("أحب البرمجة."),
    HumanMessage("I love building applications.")
]

message = model_nemotron3_nano_precise.invoke(conversation)

In [ ]:
print(message.content)

أحب بناء التطبيقات.


### 2. Streaming and chunks

Most models can stream their output content while it is being generated. By displaying output progressively, streaming significantly improves user experience, particularly for longer responses.

Calling `stream()` returns an iterator that yields output [`AIMessageChunk`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessageChunk) objects as they are produced. You can use a loop to process each chunk in real-time.

In [ ]:
chunks = []
for chunk in model_nemotron3_nano_precise.stream("what is Ramadan? keep it short, keep it simple."):
    chunks.append(chunk)
    print(chunk.text, end="", flush=True)

Ramadan is the ninth month of the Islamic lunar calendar. During it, Muslims fast from sunrise to sunset—no food, drink, or other physical needs—while also increasing prayer, charity, and reflection. The month ends with the celebration of Eid al‑Fitr.content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019c8ac4-24a9-7243-b435-06c1fabbe92f' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]


In [ ]:
for ch in chunks[50:60]:
    print(ch.content)

 of
 the
 Islamic
 lunar
 calendar
.
 During
 it
,
 Muslims


Learn more: [Streaming | LangChain](https://docs.langchain.com/oss/python/langchain/streaming).

### 3. Batch

Batching a collection of independent requests to a model can significantly improve performance and reduce costs, as the processing can be done in parallel:

In [ ]:
responses = model_nemotron3_nano_precise.batch([
    "What is the capital of Saudi Arabia?",
    "What is 2 + 8",
    "Is the sky blue or is it our perception? give a short and concise answer"
])

for response in responses:
    print(response)

content='The capital of Saudi Arabia is **Riyadh**.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 24, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 26, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 'system_fingerprint': None, 'id': 'gen-1771587230-rws1sKd4yd2Ir45jBE92', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c7ad3-dab8-7670-aebf-be1a19d73902-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 24, 'output_tokens': 38, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

In [ ]:
for i, response in enumerate(responses):
    print(response.content)
    print("="*100)

The capital of Saudi Arabia is **Riyadh**.
2 + 8 = 10.
The sky appears blue because molecules in the atmosphere scatter short‑wavelength (blue) sunlight—an objective physical effect that our visual system interprets as the color blue.


## Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

[Pydantic models](https://docs.pydantic.dev/latest/concepts/models/#basic-model-usage) provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

In [92]:
model_with_structure = model_nemotron3_nano_precise.with_structured_output(Movie)
response = model_with_structure.invoke(
    "Provide details about the movie Zootropolis"
)

In [ ]:
print("Title:", response.title)
print("Year:", response.year)
print("Director:", response.director)
print("Rating:", response.rating)

Title: Inception
Year: 2010
Director: Christopher Nolan
Rating: 8.8


Another example: **Refining Search Query**

In [ ]:
# Schema for structured output
from pydantic import BaseModel, Field


class SearchQuery(BaseModel):
    search_query: str = Field(None, description="Query that is optimized web search.")
    justification: str = Field(
        None, description="Why this query is relevant to the user's request."
    )

# Augment the LLM with schema for structured output
structured_llm = model_nemotron3_nano_precise.with_structured_output(SearchQuery)

In [ ]:
# Invoke the augmented LLM
output = structured_llm.invoke(
    "How does Calcium CT score relate to high cholesterol?"
)

In [93]:
print("search_query:", output.search_query)
print("justification:", output.justification)

search_query: Calcium CT score high cholesterol relationship atherosclerosis risk
justification: The user wants to understand the connection between a coronary artery calcium (CAC) score obtained from a CT scan and elevated blood cholesterol levels. Searching for how these two concepts interact will retrieve medical literature and clinical guidelines that explain the pathophysiology, risk stratification, and management implications.


### Benefits of Structured Output

Models are trained specifically to structure their outputs, because benefits where paramount:

- **Cost**: the model doesn't generate extra text.
- **Accuracy**: you get only what you want; no more, no less.
- **Programming**: structre can be parsed and used in programs. Example: tool calling.

Learn more: [structured responses | LangChain](https://docs.langchain.com/oss/python/langchain/structured-output)

## Key Takeawys

- Three key methods for models: invoke, stream, and batch.
- LLMs can be configured to responsd in a structured format

In [110]:
from typing import Literal
from pydantic import BaseModel, Field

class sentiment (BaseModel):
    """The sentiment of the text."""
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        ..., description="The sentiment of the text." )

    model_with_structure = model_nemotron3_nano_precise.with_structured_output(sentiment)
response = model_with_structure.invoke(
    "Kindness creates lasting joy"
)

ValueError: Unsupported function

annotation=NoneType required=True description='The sentiment of the text.'

Functions must be passed in as Dict, pydantic.BaseModel, or Callable. If they're a dict they must either be in OpenAI function format or valid JSON schema with top-level 'title' key.

In [111]:
from typing import Literal
from pydantic import BaseModel, Field

class SentimentAnalysis(BaseModel):
    """The sentiment of the text."""
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        ..., description="The sentiment of the text."
    )

model_with_structure = model_nemotron3_nano_precise.with_structured_output(SentimentAnalysis)

response = model_with_structure.invoke(
    "Kindness creates lasting joy"
)

print(response.sentiment)

positive
